In [1]:
import cftime
import pandas as pd
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
from scipy.ndimage import label
from pathlib import Path
import pyarrow as pa
from collections import Counter
import regionmask
import matplotlib.patheffects as path_effects 
import cartopy.io.shapereader as shpreader


/home/michsh/Jupyter_Env/lib64/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
chunks = pd.read_csv('/data1/michsh/CSV/HIST_derived_metrics_2.csv', chunksize=200000)

first_chunk = next(chunks)

print(first_chunk.head())

  period  AMOC  member        lat    lon  year  month  annual_frequency  \
0   HIST  1231       1  24.031414  235.0  1850     10                 1   
1   HIST  1231       1  24.031414  235.0  1851      4                 5   
2   HIST  1231       1  24.031414  235.0  1851      7                 5   
3   HIST  1231       1  24.031414  235.0  1851      9                 5   
4   HIST  1231       1  24.031414  235.0  1851     10                 5   

   monthly_frequency  event_id           start_time             end_time  \
0                  1         1  1850-10-14 00:00:00  1850-10-18 00:00:00   
1                  1         2  1851-04-13 00:00:00  1851-04-16 00:00:00   
2                  1         3  1851-07-13 00:00:00  1851-07-15 00:00:00   
3                  1         4  1851-09-06 00:00:00  1851-09-09 00:00:00   
4                  2         5  1851-10-03 00:00:00  1851-10-06 00:00:00   

   duration  mean_intensity  max_intensity  total_intensity  
0         5        0.701941   

In [3]:
import pandas as pd
import numpy as np

# ==========================================
# 1. TARGET COORDINATES (Iowa / Ames)
# ==========================================
target_lat = 42.0
target_lon = -94.0

# Convert longitude to 0-360 range if CSV uses positive longitudes
target_lon_360 = target_lon + 360 if target_lon < 0 else target_lon

# CSV Path
csv_path = '/data1/michsh/CSV/HIST_derived_metrics_2.csv'

# ==========================================
# 2. CHUNK PROCESSING WITH LOCATION FILTER
# ==========================================
chunks = pd.read_csv(csv_path, chunksize=200000)

max_value = -float('inf')
highest_row = None
highest_col_name = None

for chunk in chunks:
    # Handle longitude conventions in CSV (0-360 vs -180 to 180)
    if 'lon' in chunk.columns:
        lon_col = 'lon'
    else:
        raise KeyError("Longitude column not found in CSV.")

    lat_col = 'lat' 

    # Check coordinate system used in CSV for longitude
    sample_lon = chunk[lon_col].iloc[0]
    match_lon = target_lon_360 if sample_lon > 180 else target_lon

    # Find the nearest lat/lon present in this chunk
    lat_diff = (chunk[lat_col] - target_lat).abs()
    lon_diff = (chunk[lon_col] - match_lon).abs()
    
    # Filter for exact or nearest grid point (using small tolerance, e.g., 0.5 degrees)
    location_mask = (lat_diff < 0.5) & (lon_diff < 0.5)
    filtered_chunk = chunk[location_mask]

    if filtered_chunk.empty:
        continue

    # Identify frequency columns
    freq_cols = [col for col in filtered_chunk.columns if 'annual_frequency' in col]

    for col in freq_cols:
        # Skip if column has all NaN values
        if filtered_chunk[col].dropna().empty:
            continue

        # Find highest value in current chunk for this column at Iowa location
        chunk_max_idx = filtered_chunk[col].idxmax()
        chunk_max_val = filtered_chunk.loc[chunk_max_idx, col]

        # Update if higher value found
        if chunk_max_val > max_value:
            max_value = chunk_max_val
            highest_row = filtered_chunk.loc[chunk_max_idx]
            highest_col_name = col

# ==========================================
# 3. RESULTS
# ==========================================
if highest_row is not None:
    print(f"Target Iowa Location: Lat {target_lat}, Lon {target_lon}")
    print(f"Highest frequency found in column: '{highest_col_name}' (Value: {max_value})\n")
    print("--- Full Row Details ---")
    print(highest_row)
else:
    print("No matching data found for the target Iowa location.")

Target Iowa Location: Lat 42.0, Lon -94.0
Highest frequency found in column: 'annual_frequency' (Value: 13)

--- Full Row Details ---
period                              HIST
AMOC                                1251
member                                13
lat                            41.937173
lon                               266.25
year                                1869
month                                  4
annual_frequency                      13
monthly_frequency                      3
event_id                              67
start_time           1869-04-01 00:00:00
end_time             1869-04-04 00:00:00
duration                               4
mean_intensity                  0.932069
max_intensity                   2.241813
total_intensity                 3.728275
Name: 14411462, dtype: object


# Tick/Plotting Set-Up

In [4]:
import numpy as np

def generate_custom_ticks_1(grid_values):
    # 1. Get exact data limits
    exact_min = np.nanmin(grid_values)
    exact_max = np.nanmax(grid_values)
    
    if np.isnan(exact_min): 
        exact_min, exact_max = -7.0, 7.0
        
    # 2. Find the largest absolute value to force symmetry around 0
    max_abs = int(np.ceil(max(abs(exact_min), abs(exact_max))))
    
    # Force a minimum symmetric boundary (e.g., if max change is only 1, force a clean map)
    if max_abs == 0: 
        max_abs = 1
        
    # 3. Establish symmetric exact visual boundaries for pcolormesh (vmin / vmax)
    exact_min_sym = -max_abs
    exact_max_sym = max_abs
    
    # 4. Generate whole-number ticks from -max_abs to +max_abs
    # We step by 1 or 2 depending on how wide the span is to keep the colorbar clean
    step = 1
    custom_ticks = list(range(-max_abs, max_abs + 1, step))
    
    # 5. Create clean string labels without decimal points
    tick_labels = [f"{t}" for t in custom_ticks]
    
    return custom_ticks, tick_labels, exact_max_sym, exact_min_sym

In [5]:
import numpy as np

def generate_custom_ticks_05(grid_values):
    # 1. Get exact data limits
    exact_min = np.nanmin(grid_values)
    exact_max = np.nanmax(grid_values)
    
    if np.isnan(exact_min): exact_min, exact_max = 0.0, 2.0
    
    # 2. Round outward to the nearest 0.5 step to establish the uniform grid
    grid_start = np.floor(exact_min * 2) / 2
    grid_end = np.ceil(exact_max * 2) / 2
    
    # 3. Generate the full uniform step array (inclusive of grid_end)
    full_grid = np.arange(grid_start, grid_end + 0.1, 0.5)
    
    # 4. Slice off the fake outer boundaries to leave only the true middle ticks
    middle_ticks = list(full_grid[1:-1])
    
    # 5. Group together: [Exact Min, Middle Ticks..., Exact Max]
    custom_ticks = middle_ticks
    
    # 6. Generate text labels matching the spacing rules
    tick_labels = []
    for t in middle_ticks:
        if t % 1 == 0:
            tick_labels.append(f"{t:.1f}") # Pure whole numbers get no decimals
        else:
            tick_labels.append(f"{t:.1f}")  # Half-steps get 1 decimal place (e.g., 2.5)
            
    return custom_ticks, tick_labels, exact_max, exact_min

In [6]:
import numpy as np

def generate_custom_ticks_02(grid_values):
    # 1. Get exact data limits
    exact_max = np.nanmax(grid_values)
    
    # 2. Round outward to the nearest 0.5 step to establish the uniform grid
    grid_start = np.floor(-exact_max * 2) / 2
    grid_end = np.ceil(exact_max * 2) / 2
    
    # 3. Generate the full uniform step array (inclusive of grid_end)
    full_grid = np.arange(grid_start, grid_end + 0.1, .1)
    
    # 4. Slice off the fake outer boundaries to leave only the true middle ticks
    middle_ticks = list(full_grid[1:-1])
    
    # 5. Group together: [Exact Min, Middle Ticks..., Exact Max]
    custom_ticks = [-exact_max] + middle_ticks + [exact_max]
    
    # 6. Generate text labels matching the spacing rules
    tick_labels = []
    for t in custom_ticks:
        if t % 1 == 0:
            tick_labels.append(f"{t:.1f}") # Pure whole numbers get no decimals
        else:
            tick_labels.append(f"{t:.1f}")  # Half-steps get 1 decimal place (e.g., 2.5)
            
    return custom_ticks, tick_labels, exact_max, -exact_max

In [7]:

# --- STEP 1: Load Canada and Mexico Geometries once ---
shpfilename = shpreader.natural_earth(resolution='50m', category='cultural', name='admin_0_countries')
reader = shpreader.Reader(shpfilename)
records = reader.records()

canada_geom = None
mexico_geom = None

for record in records:
    country_name = record.attributes.get('NAME')
    if country_name == 'Canada':
        canada_geom = record.geometry
    elif country_name == 'Mexico':
        mexico_geom = record.geometry

# Load 50m Lakes Shapefile from Natural Earth
lakes_shp = shpreader.natural_earth(resolution='50m', category='physical', name='lakes')
lakes_reader = shpreader.Reader(lakes_shp)

# Names of the Great Lakes to isolate
great_lakes_names = {'Lake Superior', 'Lake Michigan', 'Lake Huron', 'Lake Erie', 'Lake Ontario'}

# Filter out only the Great Lakes geometries
great_lakes_geoms = []
for record in lakes_reader.records():
    # 'name' is the attribute key for the lake's name in Natural Earth
    lake_name = record.attributes.get('name')
    if lake_name in great_lakes_names:
        great_lakes_geoms.append(record.geometry)

# Use Natural Earth's defined regions for US States (50m resolution)
us_states = regionmask.defined_regions.natural_earth_v5_0_0.us_states_50

# Annual Frequency Calculations/Plot

In [8]:
chunks_hist = []
chunks_ftr = []

# --- STEP 1: First annual_frequency for each (AMOC, member, lat, lon, year) ---
for chunk_hist in pd.read_csv('/data1/michsh/CSV/HIST_derived_metrics_2.csv', chunksize=200000):
    grouped_chunk_hist = (
        chunk_hist
        .groupby(['AMOC', 'member', 'lat', 'lon', 'year'])['annual_frequency']
        .first()
        .reset_index()
    )
    chunks_hist.append(grouped_chunk_hist)

for chunk_ftr in pd.read_csv('/data1/michsh/CSV/FUT_derived_metrics_2.csv', chunksize=200000):
    grouped_chunk_ftr = (
        chunk_ftr
        .groupby(['AMOC', 'member', 'lat', 'lon', 'year'])['annual_frequency']
        .first()
        .reset_index()
    )
    chunks_ftr.append(grouped_chunk_ftr)

# Combine all chunks and resolve any potential edge-case duplicates across chunk boundaries
df_yearly_hist = (
    pd.concat(chunks_hist)
    .groupby(['AMOC', 'member', 'lat', 'lon', 'year'])['annual_frequency']
    .first()
    .reset_index()
)

df_yearly_ftr = (
    pd.concat(chunks_ftr)
    .groupby(['AMOC', 'member', 'lat', 'lon', 'year'])['annual_frequency']
    .first()
)

ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

In [ ]:
all_years_hist = list(range(1850, 2014+1))
all_years_ftr = list(range(2015, 2100+1))

# # Move 'year' to the columns axis to easily pad missing years with 0
# df_years_padded = df_yearly.unstack(level='year', fill_value=0)
# df_years_padded = df_years_padded.reindex(columns=all_years_hist, fill_value=0)

# # --- STEP 2: Mean across years -> one value per (AMOC, member, lat, lon) ---
# df_mean_years = df_years_padded.mean(axis=1)

# Move 'year' to the columns axis to easily pad missing years with 0
df_years_padded_hist = df_yearly_hist.unstack(level='year', fill_value=0)
df_years_padded_hist = df_years_padded_hist.reindex(columns=all_years_hist, fill_value=0)

df_years_padded_ftr = df_yearly_ftr.unstack(level='year', fill_value=0)
df_years_padded_ftr = df_years_padded_ftr.reindex(columns=all_years_ftr, fill_value=0)

# --- STEP 1: Mean across years -> Yields individual means for each of the 80 members ---
# Slicing with .mean(axis=1) collapses the 'year' columns. 
# We add .reset_index() here to transform the resulting MultiIndex Series into a flat, 
# clean DataFrame containing exactly 5 columns: ['AMOC', 'member', 'lat', 'lon', 'mean_annual_frequency']
hist_member_means_freq = (
    df_years_padded_hist
    .mean(axis=1)
    .reset_index(name='mean_annual_frequency')
)

ftr_member_means_freq = (
    df_years_padded_ftr
    .mean(axis=1)
    .reset_index(name='mean_annual_frequency')
)


# --- STEP 2: Mean across the 80 Ensemble Members -> Yields one value per spatial grid box ---
# Now we group solely by the spatial dimensions ['lat', 'lon'] to average out the 
# unique ensemble configurations ('AMOC' and 'member').
hist_ensemble_mean_frequency = (
    hist_member_means_freq
    .groupby(['lat', 'lon'])['mean_annual_frequency']
    .mean()
    .reset_index(name='mean_annual_frequency')
)

ftr_ensemble_mean_frequency = (
    ftr_member_means_freq
    .groupby(['lat', 'lon'])['mean_annual_frequency']
    .mean()
    .reset_index(name='mean_annual_frequency')
)

In [ ]:
hist_member_means_freq

In [ ]:
import numpy as np

# --- 1. Define the Hotspot Coordinates (Midwest/Iowa) ---
hotspot_lat = 44
hotspot_lon = 290

# --- 2. Handle Longitude Scale Alignment ---
# If your DataFrame longitudes were normalized to (-180 to 180),
# we must convert the target 266.5° E to -93.5° W to find a match.
target_lon_adj = hotspot_lon
if (hist_ensemble_mean_frequency['lon'] < 0).any() and hotspot_lon > 180:
    target_lon_adj = hotspot_lon - 360

# --- 3. Find the Mathematically Closest Grid Point in the CSV Data ---
# Extract all unique coordinates present in the dataset
unique_pairs = hist_ensemble_mean_frequency[['lat', 'lon']].drop_duplicates()

# Calculate Euclidean distance to the target coordinates
distances = np.sqrt(
    (unique_pairs['lat'] - hotspot_lat)**2 + 
    (unique_pairs['lon'] - target_lon_adj)**2
)
closest_idx = distances.idxmin()
exact_lat = unique_pairs.loc[closest_idx, 'lat']
exact_lon = unique_pairs.loc[closest_idx, 'lon']

# --- 4. Query the Historical and Future Frequencies ---
test_point_hist_mean = hist_ensemble_mean_frequency.loc[
    (hist_ensemble_mean_frequency['lat'] == exact_lat) & 
    (hist_ensemble_mean_frequency['lon'] == exact_lon), 
    'mean_annual_frequency'
].values[0]

test_point_ftr_mean = ftr_ensemble_mean_frequency.loc[
    (ftr_ensemble_mean_frequency['lat'] == exact_lat) & 
    (ftr_ensemble_mean_frequency['lon'] == exact_lon), 
    'mean_annual_frequency'
].values[0]

# --- 5. Print Results ---
print("--- Results for Hotspot Grid Point (Midwest) ---")
print(f"Requested Target:             ({hotspot_lat}, {hotspot_lon})")
print(f"Nearest CSV Grid Point Found: ({exact_lat:.4f}, {exact_lon:.4f})")
print("-" * 50)
print(f"CSV Map Historical Mean:      {test_point_hist_mean:.4f} events/year")
print(f"CSV Map Future Mean:          {test_point_ftr_mean:.4f} events/year")
print(f"Delta Frequency at Hotspot:   {test_point_ftr_mean - test_point_hist_mean:.4f} events/year")

In [ ]:
masked_data_arrays = []
for df in [hist_ensemble_mean_frequency, ftr_ensemble_mean_frequency]:
    # Extract coordinates
    lats = df['lat'].unique()
    lons = df['lon'].unique()
    
    # 1. Calculate the fractional overlap of each grid cell with the US States.
    # We pass the 1D lat/lon coordinates directly to the fractional mask generator
    frac_mask_3d = us_states.mask_3D_frac_approx(lons, lats, wrap_lon=True)
    
    # 2. Collapse the 'region' dimension by checking if a cell overlaps with ANY US State.
    # We set the threshold to > 0 to keep any cell that touches or sits on the border.
    any_overlap_mask = (frac_mask_3d > 0).any(dim="region")
    
    # 3. Pivot dataframe to apply spatial mask
    pivoted = df.pivot(index='lat', columns='lon', values='mean_annual_frequency')
    
    # 4. Mask the pivoted data using our fractional boolean mask
    # (Because any_overlap_mask is an xarray DataArray, we extract its values using .values)
    pivoted_masked = pivoted.where(any_overlap_mask.values)
    
    # 5. Flatten/melt back into the clean flat DataFrame format
    flat_masked = (
        pivoted_masked
        .stack(dropna=True) # Drops all grid coordinates outside CONUS
        .reset_index(name='mean_annual_frequency')
    )
    masked_data_arrays.append(flat_masked)

hist_conus_mean_masked, ftr_conus_mean_masked = masked_data_arrays

data_arrays = [hist_conus_mean_masked, ftr_conus_mean_masked]

global_min = min(hist_conus_mean_masked['mean_annual_frequency'].min(), ftr_conus_mean_masked['mean_annual_frequency'].min())
global_max = max(hist_conus_mean_masked['mean_annual_frequency'].max(), ftr_conus_mean_masked['mean_annual_frequency'].max())

combined_min_max_grid = np.array([global_min, global_max])
custom_ticks, tick_labels, exact_max, exact_min = generate_custom_ticks_05(combined_min_max_grid)

for array in data_arrays: 
    plot_target_da = array  

    if plot_target_da is hist_conus_mean_masked:
        title = "Historical Thirstwave Frequency | 80-Member Ensemble Mean"
        cb_label = "Days"
        save_name = 'Frequency_Mean_HIST'
    elif plot_target_da is ftr_conus_mean_masked:
        title = "Future Thirstwave Frequency | 80-Member Ensemble Mean"
        cb_label = "Days"
        save_name = 'Frequency_Mean_FTR'

    # This transforms the flat DataFrame into a clean 2D grid of values
    grid_data_dur = (
        plot_target_da
        .pivot(index='lat', columns='lon', values='mean_annual_frequency')
        .sort_index(ascending=True)         # Sort Latitudes monotonically
        .sort_index(axis=1, ascending=True) # Sort Longitudes monotonically
    )

    # Create Meshgrid from sorted 1D coordinates
    X, Y = np.meshgrid(grid_data_dur.columns, grid_data_dur.index)

    s_lat, n_lat = 24, 52
    w_lon, e_lon = -121, -73

    crs = ccrs.LambertConformal(central_longitude=-97, central_latitude=40)

    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=crs)

    # Set map boundaries
    ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())  

    water_color = '#a5c9eb'  # Soft pastel blue
    ax.set_facecolor(water_color)  

    im = ax.pcolormesh(X, Y, grid_data_dur.values, 
                       transform=ccrs.PlateCarree(), 
                       cmap='turbo', # <==================================================================== COLORTABLE
                       vmin=exact_min, vmax=exact_max, 
                       shading='auto',
                       zorder=1)

    if canada_geom is not None:
        ax.add_geometries([canada_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#717376', edgecolor='black', linewidth=0.65, zorder=2)
    if mexico_geom is not None:
        ax.add_geometries([mexico_geom], crs=ccrs.PlateCarree(), 
                          facecolor="#717376", edgecolor='black', linewidth=0.65, zorder=2)
    if great_lakes_geoms:
        ax.add_geometries(great_lakes_geoms, crs=ccrs.PlateCarree(),
                          facecolor=water_color, edgecolor='black', linewidth=0.65, zorder=2)

    # USA outer outline
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor=water_color, zorder=2)
    ax.coastlines('50m', linewidth=0.65, color='black', zorder=3)

    # USA outer outline
    usa_outline = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none',
        edgecolor='black',
        linewidth=0.8
    )
    ax.add_feature(usa_outline, zorder=3)

    # US State Boundaries
    states = cfeature.NaturalEarthFeature(
        category='cultural', 
        name='admin_1_states_provinces_lakes', 
        scale='50m', 
        facecolor='none'
    )
    ax.add_feature(states, edgecolor='black', linewidth=0.75, zorder=3)

    # Map features
    ax.coastlines('50m', linewidth=0.65)
    states = cfeature.NaturalEarthFeature(category='cultural', name='admin_1_states_provinces_lakes', scale='50m', facecolor='none')
    ax.add_feature(states, edgecolor='black', linewidth=0.65)

        # Drawing gridlines
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.65, 
                      color="#9d9d9d", zorder= 4, alpha=0.8)

    # Restrict labels to ONLY the bottom and left
    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = True
    gl.left_labels = True

    gl.x_inline = False
    gl.y_inline = False

    # --- ADDED: Set padding (in points) between labels and the map frame ---
    gl.xpadding = 4  # Adds space for bottom (longitude) labels
    gl.ypadding = 4  # Adds space for left (latitude) labels

    # Style the labels (removed the non-standard 'pad' key)
    gl.xlabel_style = {'size': 12, 'color': 'black', 'rotation': 15}
    gl.ylabel_style = {'size': 12, 'color': 'black'}

    # Force clean tick intervals
    gl.xlocator = mticker.FixedLocator(range(-130, -60, 10))
    gl.ylocator = mticker.FixedLocator(range(20, 55, 5))
    
    ax_box = ax.inset_axes([0.90, 0, 0.10, 0.45], zorder=5)
    ax_box.set_facecolor('white')
    ax_box.set_xticks([])
    ax_box.set_yticks([])
    for spine in ax_box.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)

    # Adjusted to end at y=0.78, leaving the top 22% of the white box for the title
    # Pushed to x=0.20 to leave plenty of room on the right side for tick labels
    cax = ax_box.inset_axes([0.10, 0.04, 0.49, 0.84]) 

    # Plot vertical colorbar (Notice: cb.ax.set_title was removed!)
    cb = plt.colorbar(im, cax=cax, orientation='vertical', ticks=custom_ticks) 

    # Style the colorbar ticks & tick labels
    cb.ax.set_yticklabels(tick_labels, fontsize=10, color='black')
    cb.ax.tick_params(direction='out', length=4, width=1)

    # --- ADDED: Text Title inside the White Box ---
    # Centered at x=0.5 (middle of the box) and y=0.89 (near the top)
    ax_box.text(
        0.5, 0.94, 
        "Events", 
        transform=ax_box.transAxes, 
        fontsize=12, 
        weight='bold', 
        color='black',
        ha='center', 
        va='center'
    )

    # Title & Layout rendering
    plt.title(title, fontsize=20, pad=13, weight='bold')
    plt.tight_layout()

    plt.savefig(save_name, dpi=300, bbox_inches='tight')
    plt.show()  

In [ ]:
data_arrays = [hist_conus_mean_masked, ftr_conus_mean_masked]

global_min = min(hist_conus_mean_masked['mean_annual_frequency'].min(), ftr_conus_mean_masked['mean_annual_frequency'].min())
global_max = max(hist_conus_mean_masked['mean_annual_frequency'].max(), ftr_conus_mean_masked['mean_annual_frequency'].max())

combined_min_max_grid = np.array([global_min, global_max])
custom_ticks, tick_labels, exact_max, exact_min = generate_custom_ticks_05(combined_min_max_grid)

for array in data_arrays: 
    plot_target_da = array  

    if plot_target_da is hist_conus_mean_masked:
        title = "Historical Thirstwave Frequency | 80-Member Ensemble Mean"
        cb_label = "Days"
        save_name = 'Frequency_Mean_HIST2.png'
    elif plot_target_da is ftr_conus_mean_masked:
        title = "Future Thirstwave Frequency | 80-Member Ensemble Mean"
        cb_label = "Days"
        save_name = 'Frequency_Mean_FTR2.png'

    # Transform flat DataFrame into a clean 2D grid
    grid_data_dur = (
        plot_target_da
        .pivot(index='lat', columns='lon', values='mean_annual_frequency')
        .sort_index(ascending=True)         # Sort Latitudes monotonically
        .sort_index(axis=1, ascending=True) # Sort Longitudes monotonically
    )

    # Create Meshgrid from sorted 1D coordinates
    X, Y = np.meshgrid(grid_data_dur.columns, grid_data_dur.index)

    s_lat, n_lat = 24, 52
    w_lon, e_lon = -121, -73

    crs = ccrs.LambertConformal(central_longitude=-97, central_latitude=40)

    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=crs)

    # Set map boundaries
    ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())  

    water_color = '#a5c9eb'  # Soft pastel blue
    ax.set_facecolor(water_color)  

    # Plot the CONUS Grid Data
    im = ax.pcolormesh(X, Y, grid_data_dur.values, 
                       transform=ccrs.PlateCarree(), 
                       cmap='turbo', 
                       vmin=exact_min, vmax=exact_max, 
                       shading='auto',
                       zorder=1)

    if canada_geom is not None:
        ax.add_geometries([canada_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#717376', edgecolor='black', linewidth=0.65, zorder=2)
    if mexico_geom is not None:
        ax.add_geometries([mexico_geom], crs=ccrs.PlateCarree(), 
                          facecolor="#717376", edgecolor='black', linewidth=0.65, zorder=2)
    if great_lakes_geoms:
        ax.add_geometries(great_lakes_geoms, crs=ccrs.PlateCarree(),
                          facecolor=water_color, edgecolor='black', linewidth=0.65, zorder=2)

    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor=water_color, zorder=2)
    ax.coastlines('50m', linewidth=0.65, color='black', zorder=3)

    # USA outer outline
    usa_outline = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none',
        edgecolor='black',
        linewidth=0.8
    )
    ax.add_feature(usa_outline, zorder=3)

    # US State Boundaries
    states = cfeature.NaturalEarthFeature(
        category='cultural', 
        name='admin_1_states_provinces_lakes', 
        scale='50m', 
        facecolor='none'
    )
    ax.add_feature(states, edgecolor='black', linewidth=0.75, zorder=3)

    # Drawing gridlines (fixed collection zorder issue)
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.65, 
                      color="#9d9d9d", alpha=0.8, zorder=4)

    # Restrict labels to ONLY the bottom and left
    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = True
    gl.left_labels = True

    gl.x_inline = False
    gl.y_inline = False

    gl.xpadding = 4  
    gl.ypadding = 4  

    # Style the labels
    gl.xlabel_style = {'size': 12, 'color': 'black', 'rotation': 15}
    gl.ylabel_style = {'size': 12, 'color': 'black'}

    # Force clean tick intervals
    gl.xlocator = mticker.FixedLocator(range(-130, -60, 10))
    gl.ylocator = mticker.FixedLocator(range(20, 55, 5))
    
    # Inset background box for colorbar
    ax_box = ax.inset_axes([0.90, 0, 0.10, 0.45], zorder=5)
    ax_box.set_facecolor('white')
    ax_box.set_xticks([])
    ax_box.set_yticks([])
    for spine in ax_box.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)

    cax = ax_box.inset_axes([0.10, 0.04, 0.49, 0.84]) 

    # Plot vertical colorbar
    cb = plt.colorbar(im, cax=cax, orientation='vertical', ticks=custom_ticks) 

    # Style the colorbar ticks & tick labels
    cb.ax.set_yticklabels(tick_labels, fontsize=10, color='black')
    cb.ax.tick_params(direction='out', length=4, width=1)

    # Text Title inside the White Box
    ax_box.text(
        0.5, 0.94, 
        "Events", 
        transform=ax_box.transAxes, 
        fontsize=12, 
        weight='bold', 
        color='black',
        ha='center', 
        va='center'
    )

    # --- ADDED: Overlay Numerical Text Values on the Map ---
    # We loop through coordinates and display non-NaN values
    for lat_val in grid_data_dur.index:
        for lon_val in grid_data_dur.columns:
            val = grid_data_dur.loc[lat_val, lon_val]
            if pd.notna(val):  # Ensure cell isn't empty/NaN
                txt = ax.text(
                    lon_val, lat_val, 
                    f"{val:.1f}",             # Format to one decimal place (e.g. 3.3)
                    transform=ccrs.PlateCarree(),
                    color='black', 
                    fontsize=6,               # Adjust size if text is overlapping
                    weight='bold',
                    ha='center', va='center',
                    zorder=6                  # Draw on top of all land/water overlays
                )
                # Adds a crisp 2px white outline to keep the text legible over any color
                txt.set_path_effects([
                    path_effects.withStroke(linewidth=2, foreground='white')
                ])

    # Title & Layout rendering
    plt.title(title, fontsize=20, pad=13, weight='bold')
    plt.tight_layout()

    plt.savefig(save_name, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
from scipy import stats

gridboxes = hist_member_means_freq[['lat', 'lon']].drop_duplicates()
results = []

for row in gridboxes.itertuples():
    lat = row.lat
    lon = row.lon

    hist_subset = hist_member_means_freq[(hist_member_means_freq.lat == lat) & (hist_member_means_freq.lon == lon)]
    ftr_subset = ftr_member_means_freq[(ftr_member_means_freq.lat == lat) & (ftr_member_means_freq.lon == lon)]

    hist_values = hist_subset['mean_annual_frequency'].values
    ftr_values = ftr_subset['mean_annual_frequency'].values

    t_stat, p_value = stats.ttest_ind(
        ftr_values,
        hist_values,
        equal_var=True,
        alternative='greater',
        nan_policy='omit'
    )
    results.append([lat, lon, p_value])

p_values_df = pd.DataFrame(results, columns=['lat', 'lon', 'p_value'])
p_values_grid = p_values_df.pivot(index='lat', columns='lon', values='p_value')

# Create p-value xarray DataArray
p_val_da = xr.DataArray(
    p_values_grid.values,
    coords=[p_values_grid.index.values, p_values_grid.columns.values],
    dims=['lat', 'lon']
)

print(p_val_da)

# 1. Pivot both ensemble means to 2D grids (lat x lon)
hist_grid = hist_ensemble_mean_frequency.pivot(index='lat', columns='lon', values='mean_annual_frequency')
ftr_grid = ftr_ensemble_mean_frequency.pivot(index='lat', columns='lon', values='mean_annual_frequency')

# 2. Calculate the Delta (Future - Historical)
# Because they share the exact same grid, pandas aligns the coordinates automatically
delta_freq_grid = ftr_grid - hist_grid

print(delta_freq_grid)

In [ ]:
# --- STEP 4: Apply Fractional Spatial Masking to Delta & p-values ---

# 1. Clean and sort the coordinates using .columns (longitude) and .index (latitude)
clean_lons = np.sort(np.unique(delta_freq_grid.columns.values))
clean_lats = np.sort(np.unique(delta_freq_grid.index.values))

# 2. Force regionmask to wrap the longitudes to [-180, 180] 
frac_mask_3d = us_states.mask_3D_frac_approx(clean_lons, clean_lats, wrap_lon=180)

# 3. Collapse the 'region' dimension
any_overlap_mask = (frac_mask_3d > 0).any(dim="region")

# 4. Apply the spatial mask to both DataArrays
# We reconstruct the 2D xarray DataArray using the clean index/columns
delta_da = xr.DataArray(
    delta_freq_grid.values,
    coords=[delta_freq_grid.index.values, delta_freq_grid.columns.values],
    dims=['lat', 'lon']
)

masked_delta_freq = delta_da.where(any_overlap_mask)
masked_p_values = p_val_da.where(any_overlap_mask)

# --- STEP 5: Calculate Dynamic Ticks on the Masked DELTA Grid ---
global_min = float(masked_delta_freq.min())
global_max = float(masked_delta_freq.max())

# Use the whole-number delta tick function to handle integer fields nicely
custom_ticks, tick_labels, exact_max, exact_min = generate_custom_ticks_1(masked_delta_freq.values)

# --- STEP 6: Run Plotting Loop ---
alphas = [0.05, 0.01]

for alpha in alphas:
    # A boolean mask where p-value < alpha AND the cell is within CONUS
    significant_mask = (

            (masked_p_values<alpha)
            &
            (masked_delta_freq>0)

    )
    
    if alpha == 0.05:
        name_alpha = '005'
    else:
        name_alpha = '001'

    # Calculate total grid cells within CONUS (ignoring NaNs)
    total_gridcells = np.sum(~np.isnan(masked_delta_freq.values)) 

    # Extract lat/lon 2D grids
    lon_grid, lat_grid = np.meshgrid(masked_delta_freq['lon'].values, masked_delta_freq['lat'].values)
    
    # Extract coordinate 1D arrays for plotting stippling
    sig_mask_np = np.nan_to_num(significant_mask.values, nan=False).astype(bool)
    significant_lons = lon_grid[sig_mask_np]
    significant_lats = lat_grid[sig_mask_np]
    
    # Calculate clean CONUS percentage
    sig_cell_count = len(significant_lons) 
    percent = (sig_cell_count / total_gridcells) * 100 if total_gridcells > 0 else 0
    print(f"Alpha {alpha} - Significant cells: {sig_cell_count}/{total_gridcells} ({percent:.2f}%)")

    # Pivot and sort coordinates of the physical delta for plotting
    grid_data_delta = (
        masked_delta_freq
        .to_dataframe(name='delta')
        .reset_index()
        .pivot(index='lat', columns='lon', values='delta')
        .sort_index(ascending=True)
        .sort_index(axis=1, ascending=True)
    )

    X, Y = np.meshgrid(grid_data_delta.columns, grid_data_delta.index)

    s_lat, n_lat = 24, 52
    w_lon, e_lon = -121, -73

    crs = ccrs.LambertConformal(central_longitude=-97, central_latitude=40)

    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=crs)

    # Set map boundaries
    ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())  

    water_color = '#a5c9eb' 
    ax.set_facecolor(water_color)  

    # Plot the background physical change (delta)
    im = ax.pcolormesh(X, Y, grid_data_delta.values, 
                       transform=ccrs.PlateCarree(), 
                       cmap='RdBu_r', 
                       vmin=exact_min, vmax=exact_max, 
                       shading='auto',
                       zorder=1)
    
    if canada_geom is not None:
        ax.add_geometries([canada_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#d3d3d3', edgecolor='black', linewidth=0.8, zorder=2)
    if mexico_geom is not None:
        ax.add_geometries([mexico_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#d3d3d3', edgecolor='black', linewidth=0.8, zorder=2)
    if great_lakes_geoms:
        ax.add_geometries(great_lakes_geoms, crs=ccrs.PlateCarree(),
                          facecolor=water_color, edgecolor='black', linewidth=0.75, zorder=2)
        
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor=water_color, zorder=2)
    ax.coastlines('50m', linewidth=0.8, color='black', zorder=4)

    # USA outer outline
    usa_outline = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none',
        edgecolor='black',
        linewidth=0.8,
        zorder=4
    )
    ax.add_feature(usa_outline, zorder=3)

    # US State Boundaries
    states = cfeature.NaturalEarthFeature(
        category='cultural', 
        name='admin_1_states_provinces_lakes', 
        scale='50m', 
        facecolor='none'
    )
    ax.add_feature(states, edgecolor='black', linewidth=0.65, zorder=4)

    # Drawing gridlines (fixed zorder issue)
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.65, 
                      color="#9d9d9d", alpha=0.8, zorder=5)

    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = True
    gl.left_labels = True
    gl.x_inline = False
    gl.y_inline = False
    gl.xpadding = 4  
    gl.ypadding = 4  

    gl.xlabel_style = {'size': 12, 'color': 'black', 'rotation': 15}
    gl.ylabel_style = {'size': 12, 'color': 'black'}

    gl.xlocator = mticker.FixedLocator(range(-130, -60, 10))
    gl.ylocator = mticker.FixedLocator(range(20, 55, 5))

    # Colorbar Box Setup
    ax_box = ax.inset_axes([0.90, 0, 0.10, 0.45], zorder=6)
    ax_box.set_facecolor('white')
    ax_box.set_xticks([])
    ax_box.set_yticks([])
    for spine in ax_box.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)

    cax = ax_box.inset_axes([0.10, 0.04, 0.49, 0.84]) 
    cb = plt.colorbar(im, cax=cax, orientation='vertical', ticks=custom_ticks) 

    cb.ax.set_yticklabels(tick_labels, fontsize=10, color='black')
    cb.ax.tick_params(direction='out', length=4, width=1)

    # Text Title inside the Colorbar Box
    ax_box.text(
        0.5, 0.94, 
        'Δ Events', 
        transform=ax_box.transAxes, 
        fontsize=12, 
        weight='bold', 
        color='black',
        ha='center', 
        va='center'
    )

    # --- OVERLAY DELTA MARKERS (Stippling) ---
    # Plots the open black circles only on significant CONUS points
    ax.scatter(significant_lons, significant_lats, 
               marker='o', facecolors='none', edgecolors='black',
               s=15,          
               alpha=0.5,      
               transform=ccrs.PlateCarree(),
               zorder=1.5  # Kept above map boundaries, below gridlines
               )

    # Update title dynamically to include the computed CONUS percentage
    plt.title(f'Change in Mean Thirstwave Frequency | o = p < {alpha} ({percent:.1f}% of CONUS)', 
              fontsize=16, pad=13, weight='bold')
    plt.tight_layout()

    plt.savefig(f'Frequency_Mean_Delta_{name_alpha}', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# --- STEP 1: Split the Future Padded DataFrame into Halves ---
years_f1 = list(range(2015, 2057+1)) # 43 years
years_f2 = list(range(2058, 2100+1)) # 43 years

df_years_f1 = df_years_padded_ftr[years_f1]
df_years_f2 = df_years_padded_ftr[years_f2]

# --- STEP 2: Compute Member-Level Means ---
f1_member_means = df_years_f1.mean(axis=1).reset_index(name='mean_ann_freq')
f2_member_means = df_years_f2.mean(axis=1).reset_index(name='mean_ann_freq')

# --- STEP 3: Grid-by-Grid t-test (Late vs. Mid Century) ---
gridboxes_split = f1_member_means[['lat', 'lon']].drop_duplicates()
split_results = []

for row in gridboxes_split.itertuples():
    f1_vals = f1_member_means[(f1_member_means.lat == row.lat) & (f1_member_means.lon == row.lon)]['mean_ann_freq'].values
    f2_vals = f2_member_means[(f2_member_means.lat == row.lat) & (f2_member_means.lon == row.lon)]['mean_ann_freq'].values

    # Testing if late century (f2) frequency is significantly greater than mid-century (f1)
    t_stat, p_value = stats.ttest_ind(f2_vals, f1_vals, equal_var=True, alternative='greater', nan_policy='omit')
    split_results.append([row.lat, row.lon, p_value])

split_p_values_df = pd.DataFrame(split_results, columns=['lat', 'lon', 'p_value'])
split_p_values_grid = split_p_values_df.pivot(index='lat', columns='lon', values='p_value')
split_p_val_da = xr.DataArray(split_p_values_grid.values, coords=[split_p_values_grid.index.values, split_p_values_grid.columns.values], dims=['lat', 'lon'])

# --- STEP 4: Calculate Late-Century Acceleration Delta ---
f1_ensemble_mean = f1_member_means.groupby(['lat', 'lon'])['mean_ann_freq'].mean().reset_index(name='mean_ann_freq')
f2_ensemble_mean = f2_member_means.groupby(['lat', 'lon'])['mean_ann_freq'].mean().reset_index(name='mean_ann_freq')

f1_grid = f1_ensemble_mean.pivot(index='lat', columns='lon', values='mean_ann_freq')
f2_grid = f2_ensemble_mean.pivot(index='lat', columns='lon', values='mean_ann_freq')
delta_split_grid = f2_grid - f1_grid

# --- STEP 5: Apply Masking ---
delta_split_da = xr.DataArray(delta_split_grid.values, coords=[delta_split_grid.index.values, delta_split_grid.columns.values], dims=['lat', 'lon'])
masked_delta_split = delta_split_da.where(any_overlap_mask)
masked_split_p_values = split_p_val_da.where(any_overlap_mask)

# ========================= F1 - Hist =========================

gridboxes_hist_f1 = hist_member_means_freq[['lat', 'lon']].drop_duplicates()
hist_f1_results = []

for row in gridboxes_hist_f1.itertuples():
    hist_vals = hist_member_means_freq[(hist_member_means_freq.lat == row.lat) & (hist_member_means_freq.lon == row.lon)]['mean_annual_frequency'].values
    f1_vals = f1_member_means[(f1_member_means.lat == row.lat) & (f1_member_means.lon == row.lon)]['mean_ann_freq'].values

    # Alternative='greater' tests if mid-century (f1) is significantly greater than historical (hist)
    t_stat, p_value = stats.ttest_ind(f1_vals, hist_vals, equal_var=True, alternative='greater', nan_policy='omit')
    hist_f1_results.append([row.lat, row.lon, p_value])

# Convert t-test results into a clean 2D xarray DataArray
hist_f1_p_values_df = pd.DataFrame(hist_f1_results, columns=['lat', 'lon', 'p_value'])
hist_f1_p_values_grid = hist_f1_p_values_df.pivot(index='lat', columns='lon', values='p_value')

hist_f1_p_val_da = xr.DataArray(
    hist_f1_p_values_grid.values, 
    coords=[hist_f1_p_values_grid.index.values, hist_f1_p_values_grid.columns.values], 
    dims=['lat', 'lon']
)

# Collapse the member dimensions down to an ensemble mean per gridbox
hist_ensemble_mean = hist_member_means_freq.groupby(['lat', 'lon'])['mean_annual_frequency'].mean().reset_index(name='mean_annual_frequency')
f1_ensemble_mean = f1_member_means.groupby(['lat', 'lon'])['mean_ann_freq'].mean().reset_index(name='mean_ann_freq')

hist_grid = hist_ensemble_mean.pivot(index='lat', columns='lon', values='mean_annual_frequency')
f1_grid = f1_ensemble_mean.pivot(index='lat', columns='lon', values='mean_ann_freq')

# Mid-Century Delta: F1 minus Historical
delta_f1_hist = f1_grid - hist_grid

delta_hist_f1_da = xr.DataArray(
    delta_f1_hist.values, 
    coords=[delta_f1_hist.index.values, delta_f1_hist.columns.values], 
    dims=['lat', 'lon']
)

masked_delta_hist_f1 = delta_hist_f1_da.where(any_overlap_mask)
masked_hist_f1_p_values = hist_f1_p_val_da.where(any_overlap_mask)

In [ ]:
# =====================================================================
# --- STEP 1: Compute and Mask Spatial Ensemble Averages for Both Halves ---
# =====================================================================

# Compute the ensemble spatial means from the member-level means calculated earlier
f1_ensemble_mean_freq = (
    f1_member_means
    .groupby(['lat', 'lon'])['mean_ann_freq']
    .mean()
    .reset_index(name='mean_annual_frequency')
)

f2_ensemble_mean_freq = (
    f2_member_means
    .groupby(['lat', 'lon'])['mean_ann_freq']
    .mean()
    .reset_index(name='mean_annual_frequency')
)

# Run the fractional CONUS masking over both new future subsets
masked_split_arrays = []
for df in [f1_ensemble_mean_freq, f2_ensemble_mean_freq]:
    lats = df['lat'].unique()
    lons = df['lon'].unique()
    
    # 1. Calculate the fractional overlap
    frac_mask_3d = us_states.mask_3D_frac_approx(lons, lats, wrap_lon=True)
    any_overlap_mask = (frac_mask_3d > 0).any(dim="region")
    
    # 2. Pivot and apply the spatial mask
    pivoted = df.pivot(index='lat', columns='lon', values='mean_annual_frequency')
    pivoted_masked = pivoted.where(any_overlap_mask.values)
    
    # 3. Flatten/melt back into clean flat DataFrames
    flat_masked = (
        pivoted_masked
        .stack(dropna=True)
        .reset_index(name='mean_annual_frequency')
    )
    masked_split_arrays.append(flat_masked)

f1_conus_mean_masked, f2_conus_mean_masked = masked_split_arrays


# =====================================================================
# --- STEP 2: Configure Global Colorbar Boundaries & Mapping Loop ---
# =====================================================================

# Combine data arrays to generate shared color ticks for true side-by-side comparison
data_arrays_split = [f1_conus_mean_masked, f2_conus_mean_masked]

global_min_split = min(f1_conus_mean_masked['mean_annual_frequency'].min(), f2_conus_mean_masked['mean_annual_frequency'].min())
global_max_split = max(f1_conus_mean_masked['mean_annual_frequency'].max(), f2_conus_mean_masked['mean_annual_frequency'].max())

combined_min_max_split = np.array([global_min_split, global_max_split])
# Note: Using your custom whole or decimal tick generator for absolute frequency scales
custom_ticks, tick_labels, exact_max, exact_min = generate_custom_ticks_05(combined_min_max_split)

for array in data_arrays_split: 
    plot_target_da = array  

    if plot_target_da is f1_conus_mean_masked:
        title = "Mid-Century Thirstwave Frequency (2015–2057) | 80-Member Mean"
        save_name = 'Frequency_Mean_FUT_Half1'
    elif plot_target_da is f2_conus_mean_masked:
        title = "Late-Century Thirstwave Frequency (2058–2100) | 80-Member Mean"
        save_name = 'Frequency_Mean_FUT_Half2'

    # Transform flat DataFrame into clean 2D grid
    grid_data_dur = (
        plot_target_da
        .pivot(index='lat', columns='lon', values='mean_annual_frequency')
        .sort_index(ascending=True)         
        .sort_index(axis=1, ascending=True) 
    )

    X, Y = np.meshgrid(grid_data_dur.columns, grid_data_dur.index)

    s_lat, n_lat = 24, 52
    w_lon, e_lon = -121, -73

    crs = ccrs.LambertConformal(central_longitude=-97, central_latitude=40)

    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=crs)
    ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())  

    water_color = '#a5c9eb'  
    ax.set_facecolor(water_color)  

    im = ax.pcolormesh(X, Y, grid_data_dur.values, 
                       transform=ccrs.PlateCarree(), 
                       cmap='turbo', 
                       vmin=exact_min, vmax=exact_max, 
                       shading='auto',
                       zorder=1)

    # Add background geography geometries
    if canada_geom is not None:
        ax.add_geometries([canada_geom], crs=ccrs.PlateCarree(), facecolor='#717376', edgecolor='black', linewidth=0.65, zorder=2)
    if mexico_geom is not None:
        ax.add_geometries([mexico_geom], crs=ccrs.PlateCarree(), facecolor="#717376", edgecolor='black', linewidth=0.65, zorder=2)
    if great_lakes_geoms:
        ax.add_geometries(great_lakes_geoms, crs=ccrs.PlateCarree(), facecolor=water_color, edgecolor='black', linewidth=0.65, zorder=2)

    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor=water_color, zorder=2)
    ax.coastlines('50m', linewidth=0.65, color='black', zorder=3)

    usa_outline = cfeature.NaturalEarthFeature(category='cultural', name='admin_0_boundary_lines_land', scale='50m', facecolor='none', edgecolor='black', linewidth=0.8)
    ax.add_feature(usa_outline, zorder=3)

    states = cfeature.NaturalEarthFeature(category='cultural', name='admin_1_states_provinces_lakes', scale='50m', facecolor='none')
    ax.add_feature(states, edgecolor='black', linewidth=0.75, zorder=3)

    # Gridlines and ticks layout
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.65, color="#9d9d9d", zorder=4, alpha=0.8)
    gl.top_labels = gl.right_labels = gl.x_inline = gl.y_inline = False
    gl.bottom_labels = gl.left_labels = True
    gl.xpadding = gl.ypadding = 4
    gl.xlabel_style, gl.ylabel_style = {'size': 12, 'color': 'black', 'rotation': 15}, {'size': 12, 'color': 'black'}
    gl.xlocator = mticker.FixedLocator(range(-130, -60, 10))
    gl.ylocator = mticker.FixedLocator(range(20, 55, 5))
    
    # Render the white custom inline colorbar container box
    ax_box = ax.inset_axes([0.92, 0, 0.10, 0.45], zorder=5)
    ax_box.set_facecolor('white')
    ax_box.set_xticks([]); ax_box.set_yticks([])
    for spine in ax_box.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)

    # Left pos, bottom pos, expanded width for tick clearance, height
    cax = ax_box.inset_axes([0.15, 0.04, 0.75, 0.84]) 
    cb = plt.colorbar(im, cax=cax, orientation='vertical', ticks=custom_ticks) 
    cb.ax.set_yticklabels(tick_labels, fontsize=10, color='black')
    cb.ax.tick_params(direction='out', length=4, width=1)

    ax_box.text(0.5, 0.94, "Events/yr", transform=ax_box.transAxes, fontsize=10, weight='bold', color='black', ha='center', va='center')

    plt.title(title, fontsize=16, pad=13, weight='bold')
    plt.tight_layout()
    plt.savefig(save_name, dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# --- STEP 4: Apply Fractional Spatial Masking to Delta & p-values ---

# 1. Clean and sort the coordinates using .columns (longitude) and .index (latitude)
clean_lons = np.sort(np.unique(delta_split_grid.columns.values))
clean_lats = np.sort(np.unique(delta_split_grid.index.values))

# 2. Force regionmask to wrap the longitudes to [-180, 180] 
frac_mask_3d = us_states.mask_3D_frac_approx(clean_lons, clean_lats, wrap_lon=180)

# 3. Collapse the 'region' dimension
any_overlap_mask = (frac_mask_3d > 0).any(dim="region")

# 4. Apply the spatial mask to both DataArrays
# We reconstruct the 2D xarray DataArray using the clean index/columns
delta_da = xr.DataArray(
    delta_split_grid.values,
    coords=[delta_split_grid.index.values, delta_split_grid.columns.values],
    dims=['lat', 'lon']
)

masked_delta_freq = delta_split_da.where(any_overlap_mask)
masked_p_values = p_val_da.where(any_overlap_mask)

# --- STEP 5: Calculate Dynamic Ticks on the Masked DELTA Grid ---
global_min = float(masked_delta_freq.min())
global_max = float(masked_delta_freq.max())

# Use the whole-number delta tick function to handle integer fields nicely
custom_ticks, tick_labels, exact_max, exact_min = generate_custom_ticks_1(masked_delta_freq.values)

# --- STEP 6: Run Plotting Loop ---
alphas = [0.05, 0.01]

for alpha in alphas:
    # A boolean mask where p-value < alpha AND the cell is within CONUS
    significant_mask = (

            (masked_p_values<alpha)
            &
            (masked_delta_freq>0)

    )
    
    if alpha == 0.05:
        name_alpha = '005'
    else:
        name_alpha = '001'

    # Calculate total grid cells within CONUS (ignoring NaNs)
    total_gridcells = np.sum(~np.isnan(masked_delta_freq.values)) 

    # Extract lat/lon 2D grids
    lon_grid, lat_grid = np.meshgrid(masked_delta_freq['lon'].values, masked_delta_freq['lat'].values)
    
    # Extract coordinate 1D arrays for plotting stippling
    sig_mask_np = np.nan_to_num(significant_mask.values, nan=False).astype(bool)
    significant_lons = lon_grid[sig_mask_np]
    significant_lats = lat_grid[sig_mask_np]
    
    # Calculate clean CONUS percentage
    sig_cell_count = len(significant_lons) 
    percent = (sig_cell_count / total_gridcells) * 100 if total_gridcells > 0 else 0
    print(f"Alpha {alpha} - Significant cells: {sig_cell_count}/{total_gridcells} ({percent:.2f}%)")

    # Pivot and sort coordinates of the physical delta for plotting
    grid_data_delta = (
        masked_delta_freq
        .to_dataframe(name='delta')
        .reset_index()
        .pivot(index='lat', columns='lon', values='delta')
        .sort_index(ascending=True)
        .sort_index(axis=1, ascending=True)
    )

    X, Y = np.meshgrid(grid_data_delta.columns, grid_data_delta.index)

    s_lat, n_lat = 24, 52
    w_lon, e_lon = -121, -73

    crs = ccrs.LambertConformal(central_longitude=-97, central_latitude=40)

    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=crs)

    # Set map boundaries
    ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())  

    water_color = '#a5c9eb' 
    ax.set_facecolor(water_color)  

    # Plot the background physical change (delta)
    im = ax.pcolormesh(X, Y, grid_data_delta.values, 
                       transform=ccrs.PlateCarree(), 
                       cmap='RdBu_r', 
                       vmin=exact_min, vmax=exact_max, 
                       shading='auto',
                       zorder=1)
    
    if canada_geom is not None:
        ax.add_geometries([canada_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#d3d3d3', edgecolor='black', linewidth=0.8, zorder=2)
    if mexico_geom is not None:
        ax.add_geometries([mexico_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#d3d3d3', edgecolor='black', linewidth=0.8, zorder=2)
    if great_lakes_geoms:
        ax.add_geometries(great_lakes_geoms, crs=ccrs.PlateCarree(),
                          facecolor=water_color, edgecolor='black', linewidth=0.75, zorder=2)
        
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor=water_color, zorder=2)
    ax.coastlines('50m', linewidth=0.8, color='black', zorder=4)

    # USA outer outline
    usa_outline = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none',
        edgecolor='black',
        linewidth=0.8,
        zorder=4
    )
    ax.add_feature(usa_outline, zorder=3)

    # US State Boundaries
    states = cfeature.NaturalEarthFeature(
        category='cultural', 
        name='admin_1_states_provinces_lakes', 
        scale='50m', 
        facecolor='none'
    )
    ax.add_feature(states, edgecolor='black', linewidth=0.65, zorder=4)

    # Drawing gridlines (fixed zorder issue)
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.65, 
                      color="#9d9d9d", alpha=0.8, zorder=5)

    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = True
    gl.left_labels = True
    gl.x_inline = False
    gl.y_inline = False
    gl.xpadding = 4  
    gl.ypadding = 4  

    gl.xlabel_style = {'size': 12, 'color': 'black', 'rotation': 15}
    gl.ylabel_style = {'size': 12, 'color': 'black'}

    gl.xlocator = mticker.FixedLocator(range(-130, -60, 10))
    gl.ylocator = mticker.FixedLocator(range(20, 55, 5))

    # Colorbar Box Setup
    ax_box = ax.inset_axes([0.90, 0, 0.10, 0.45], zorder=6)
    ax_box.set_facecolor('white')
    ax_box.set_xticks([])
    ax_box.set_yticks([])
    for spine in ax_box.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)

    cax = ax_box.inset_axes([0.10, 0.04, 0.49, 0.84]) 
    cb = plt.colorbar(im, cax=cax, orientation='vertical', ticks=custom_ticks) 

    cb.ax.set_yticklabels(tick_labels, fontsize=10, color='black')
    cb.ax.tick_params(direction='out', length=4, width=1)

    # Text Title inside the Colorbar Box
    ax_box.text(
        0.5, 0.94, 
        'Δ Events', 
        transform=ax_box.transAxes, 
        fontsize=12, 
        weight='bold', 
        color='black',
        ha='center', 
        va='center'
    )

    # --- OVERLAY DELTA MARKERS (Stippling) ---
    # Plots the open black circles only on significant CONUS points
    ax.scatter(significant_lons, significant_lats, 
               marker='o', facecolors='none', edgecolors='black',
               s=15,          
               alpha=0.5,      
               transform=ccrs.PlateCarree(),
               zorder=1.5  # Kept above map boundaries, below gridlines
               )

    # Update title dynamically to include the computed CONUS percentage
    plt.title(f'Change in Mean Thirstwave Frequency | o = p < {alpha} ({percent:.1f}% of CONUS)', 
              fontsize=16, pad=13, weight='bold')
    plt.tight_layout()

    plt.savefig(f'Frequency_Mean_Delta_{name_alpha}', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# --- STEP 5: Calculate Dynamic Ticks on the Masked F1-HIST DELTA Grid ---
# Generate symmetric whole-number ticks tailored for integer delta frequencies
custom_ticks, tick_labels, exact_max, exact_min = generate_integer_delta_ticks(masked_delta_hist_f1.values)

# --- STEP 6: Run Plotting Loop for Mid-Century Delta Significant Testing ---
alphas = [0.05, 0.01]

for alpha in alphas:
    # A boolean mask where p-value < alpha AND the mid-century delta is positive
    significant_mask = (
        (masked_hist_f1_p_values < alpha) & 
        (masked_delta_hist_f1 > 0)
    )
    
    name_alpha = '005' if alpha == 0.05 else '001'

    # Calculate total valid grid cells within the CONUS domain (ignoring spatial NaNs)
    total_gridcells = np.sum(~np.isnan(masked_delta_hist_f1.values)) 

    # Extract lat/lon 2D grids for isolating coordinates
    lon_grid, lat_grid = np.meshgrid(masked_delta_hist_f1['lon'].values, masked_delta_hist_f1['lat'].values)
    
    # Isolate coordinate points that meet the significance threshold for stippling overlays
    sig_mask_np = np.nan_to_num(significant_mask.values, nan=False).astype(bool)
    significant_lons = lon_grid[sig_mask_np]
    significant_lats = lat_grid[sig_mask_np]
    
    # Calculate the precise CONUS spatial significance percentage
    sig_cell_count = len(significant_lons) 
    percent = (sig_cell_count / total_gridcells) * 100 if total_gridcells > 0 else 0
    print(f"Alpha {alpha} (F1-HIST) - Significant cells: {sig_cell_count}/{total_gridcells} ({percent:.2f}%)")

    # Pivot, clean, and sort the spatial coordinate grid for pcolormesh mapping
    grid_data_delta = (
        masked_delta_hist_f1
        .to_dataframe(name='delta')
        .reset_index()
        .pivot(index='lat', columns='lon', values='delta')
        .sort_index(ascending=True)
        .sort_index(axis=1, ascending=True)
    )

    X, Y = np.meshgrid(grid_data_delta.columns, grid_data_delta.index)

    # Establish spatial frame parameters
    s_lat, n_lat = 24, 52
    w_lon, e_lon = -121, -73

    crs = ccrs.LambertConformal(central_longitude=-97, central_latitude=40)

    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=crs)
    ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())  

    water_color = '#a5c9eb' 
    ax.set_facecolor(water_color)  

    # Plot the background physical distribution delta (F1 - HIST)
    im = ax.pcolormesh(X, Y, grid_data_delta.values, 
                       transform=ccrs.PlateCarree(), 
                       cmap='RdBu_r', 
                       vmin=exact_min, vmax=exact_max, 
                       shading='auto',
                       zorder=1)
    
    # Apply baseline geography and masking layers
    if canada_geom is not None:
        ax.add_geometries([canada_geom], crs=ccrs.PlateCarree(), facecolor='#d3d3d3', edgecolor='black', linewidth=0.8, zorder=2)
    if mexico_geom is not None:
        ax.add_geometries([mexico_geom], crs=ccrs.PlateCarree(), facecolor='#d3d3d3', edgecolor='black', linewidth=0.8, zorder=2)
    if great_lakes_geoms:
        ax.add_geometries(great_lakes_geoms, crs=ccrs.PlateCarree(), facecolor=water_color, edgecolor='black', linewidth=0.75, zorder=2)
        
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor=water_color, zorder=2)
    ax.coastlines('50m', linewidth=0.8, color='black', zorder=4)

    usa_outline = cfeature.NaturalEarthFeature(category='cultural', name='admin_0_boundary_lines_land', scale='50m', facecolor='none', edgecolor='black', linewidth=0.8, zorder=4)
    ax.add_feature(usa_outline, zorder=3)

    states = cfeature.NaturalEarthFeature(category='cultural', name='admin_1_states_provinces_lakes', scale='50m', facecolor='none')
    ax.add_feature(states, edgecolor='black', linewidth=0.65, zorder=4)

    # Set up geographic map coordinate gridlines
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.65, color="#9d9d9d", alpha=0.8, zorder=5)
    gl.top_labels = gl.right_labels = gl.x_inline = gl.y_inline = False
    gl.bottom_labels = gl.left_labels = True
    gl.xpadding = gl.ypadding = 4  

    gl.xlabel_style = {'size': 12, 'color': 'black', 'rotation': 15}
    gl.ylabel_style = {'size': 12, 'color': 'black'}
    gl.xlocator = mticker.FixedLocator(range(-130, -60, 10))
    gl.ylocator = mticker.FixedLocator(range(20, 55, 5))

    # Colorbar Inline Container Setup (Shifted to x=0.92 to prevent frame overlapping)
    ax_box = ax.inset_axes([0.92, 0, 0.10, 0.45], zorder=6)
    ax_box.set_facecolor('white')
    ax_box.set_xticks([]); ax_box.set_yticks([])
    for spine in ax_box.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)

    # Expanded width to 0.75 inside container to guarantee whole numbers render flawlessly
    cax = ax_box.inset_axes([0.15, 0.04, 0.75, 0.84]) 
    cb = plt.colorbar(im, cax=cax, orientation='vertical', ticks=custom_ticks) 
    cb.ax.set_yticklabels(tick_labels, fontsize=10, color='black')
    cb.ax.tick_params(direction='out', length=4, width=1)

    ax_box.text(0.5, 0.94, 'Δ Events', transform=ax_box.transAxes, fontsize=11, weight='bold', color='black', ha='center', va='center')

    # --- OVERLAY STIPPLING FOR SIGNIFICANT GRID CELLS ---
    ax.scatter(significant_lons, significant_lats, 
               marker='o', facecolors='none', edgecolors='black',
               s=15, alpha=0.5, transform=ccrs.PlateCarree(),
               zorder=1.5)

    # Dynamic title setting showing mid-century change context and calculated threshold metrics
    plt.title(f'Mid-Century Frequency Delta (F1 - HIST) | o = p < {alpha} ({percent:.1f}% of CONUS)', 
              fontsize=15, pad=13, weight='bold')
    plt.tight_layout()

    plt.savefig(f'Frequency_F1_Hist_Delta_{name_alpha}', dpi=300, bbox_inches='tight')
    plt.show()

# Duration Calculations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Load data in chunks and gather all events ---
# We keep only relevant columns to save memory. 
# Do not use .first() here so we collect *every* event's duration.
cols_to_keep = ['AMOC', 'member', 'lat', 'lon', 'duration']

hist_dur_chunks = []
ftr_dur_chunks = []

for chunk in pd.read_csv('/data1/michsh/CSV/HIST_derived_metrics_2.csv', chunksize=200000, usecols=cols_to_keep):
    # Optional: Filter out NaN or zero-duration rows if they exist
    chunk = chunk[chunk['duration'] > 0]
    hist_dur_chunks.append(chunk)

for chunk in pd.read_csv('/data1/michsh/CSV/FUT_derived_metrics_2.csv', chunksize=200000, usecols=cols_to_keep):
    # Optional: Filter out NaN or zero-duration rows if they exist
    chunk = chunk[chunk['duration'] > 0]
    ftr_dur_chunks.append(chunk)

# Combine all chunks into one DataFrame of active thirstwave/event durations
hist_df_events = pd.concat(hist_dur_chunks, ignore_index=True)
ftr_df_events = pd.concat(ftr_dur_chunks, ignore_index=True)

# --- 2. Calculate Mean Duration per Grid Box for Each Ensemble Member ---
# This groups by the spatial coordinates AND the unique ensemble identifiers,
# yielding the mean event duration for each of the 80 members.
hist_member_means = (
    hist_df_events
    .groupby(['lat', 'lon', 'AMOC', 'member'])['duration']
    .mean()
    .reset_index()
)

ftr_member_means = (
    ftr_df_events
    .groupby(['lat', 'lon', 'AMOC', 'member'])['duration']
    .mean()
    .reset_index()
)

# --- 3. Calculate the Mean Across the 80 Ensemble Members ---
# Now we average over the 'AMOC' and 'member' dimensions to get one value per grid box.
hist_ensemble_mean_duration = (
    hist_member_means
    .groupby(['lat', 'lon'])['duration']
    .mean()
    .reset_index(name='mean_dur')
)

ftr_ensemble_mean_duration = (
    ftr_member_means
    .groupby(['lat', 'lon'])['duration']
    .mean()
    .reset_index(name='mean_dur')
)

In [ ]:
masked_data_arrays = []
for df in [hist_ensemble_mean_duration, ftr_ensemble_mean_duration]:
    # Extract coordinates
    lats = df['lat'].unique()
    lons = df['lon'].unique()
    
    # 1. Calculate the fractional overlap of each grid cell with the US States.
    # We pass the 1D lat/lon coordinates directly to the fractional mask generator
    frac_mask_3d = us_states.mask_3D_frac_approx(lons, lats, wrap_lon=True)
    
    # 2. Collapse the 'region' dimension by checking if a cell overlaps with ANY US State.
    # We set the threshold to > 0 to keep any cell that touches or sits on the border.
    any_overlap_mask = (frac_mask_3d > 0).any(dim="region")
    
    # 3. Pivot dataframe to apply spatial mask
    pivoted = df.pivot(index='lat', columns='lon', values='mean_dur')
    
    # 4. Mask the pivoted data using our fractional boolean mask
    # (Because any_overlap_mask is an xarray DataArray, we extract its values using .values)
    pivoted_masked = pivoted.where(any_overlap_mask.values)
    
    # 5. Flatten/melt back into the clean flat DataFrame format
    flat_masked = (
        pivoted_masked
        .stack(dropna=True) # Drops all grid coordinates outside CONUS
        .reset_index(name='mean_dur')
    )
    masked_data_arrays.append(flat_masked)

hist_ensemble_mean_duration_masked, ftr_ensemble_mean_duration_masked = masked_data_arrays

# Reassign masked datasets
hist_conus_mean_masked, ftr_conus_mean_masked = masked_data_arrays

data_arrays = [hist_ensemble_mean_duration, ftr_ensemble_mean_duration]

global_min = min(hist_conus_mean_masked['mean_dur'].min(), ftr_conus_mean_masked['mean_dur'].min())
global_max = max(hist_conus_mean_masked['mean_dur'].max(), ftr_conus_mean_masked['mean_dur'].max())

combined_min_max_grid = np.array([global_min, global_max])
custom_ticks, tick_labels, exact_max, exact_min = generate_custom_ticks_05(combined_min_max_grid)

for array in data_arrays: 
    plot_target_da = array  

    if plot_target_da is hist_ensemble_mean_duration:
        title = "Historical Thirstwave Duration | 80-Member Ensemble Mean"
        cb_label = "Days"
        save_name = 'Duration_Mean_HIST'
    elif plot_target_da is ftr_ensemble_mean_duration:
        title = "Future Thirstwave Duration | 80-Member Ensemble Mean"
        cb_label = "Days"
        save_name = 'Duration_Mean_FTR'

    # This transforms the flat DataFrame into a clean 2D grid of values
    grid_data_dur = (
        plot_target_da
        .pivot(index='lat', columns='lon', values='mean_dur')
        .sort_index(ascending=True)         # Sort Latitudes monotonically
        .sort_index(axis=1, ascending=True) # Sort Longitudes monotonically
    )

    # Create Meshgrid from sorted 1D coordinates
    X, Y = np.meshgrid(grid_data_dur.columns, grid_data_dur.index)

    s_lat, n_lat = 24, 52
    w_lon, e_lon = -121, -73

    crs = ccrs.LambertConformal(central_longitude=-97, central_latitude=40)

    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=crs)

    # Set map boundaries
    ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())  

    water_color = '#a5c9eb'  # Soft pastel blue
    ax.set_facecolor(water_color)  

    im = ax.pcolormesh(X, Y, grid_data_dur.values, 
                       transform=ccrs.PlateCarree(), 
                       cmap='turbo', 
                       vmin=exact_min, vmax=exact_max, 
                       shading='auto',
                       zorder=1)

    if canada_geom is not None:
        ax.add_geometries([canada_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#717376', edgecolor='black', linewidth=0.65, zorder=2)
    if mexico_geom is not None:
        ax.add_geometries([mexico_geom], crs=ccrs.PlateCarree(), 
                          facecolor="#717376", edgecolor='black', linewidth=0.65, zorder=2)
    if great_lakes_geoms:
        ax.add_geometries(great_lakes_geoms, crs=ccrs.PlateCarree(),
                          facecolor=water_color, edgecolor='black', linewidth=0.65, zorder=2)

    # USA outer outline
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor=water_color, zorder=2)
    ax.coastlines('50m', linewidth=0.65, color='black', zorder=3)

    # USA outer outline
    usa_outline = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none',
        edgecolor='black',
        linewidth=0.8
    )
    ax.add_feature(usa_outline, zorder=3)

    # US State Boundaries
    states = cfeature.NaturalEarthFeature(
        category='cultural', 
        name='admin_1_states_provinces_lakes', 
        scale='50m', 
        facecolor='none'
    )
    ax.add_feature(states, edgecolor='black', linewidth=0.75, zorder=3)

    # Map features
    ax.coastlines('50m', linewidth=0.65)
    states = cfeature.NaturalEarthFeature(category='cultural', name='admin_1_states_provinces_lakes', scale='50m', facecolor='none')
    ax.add_feature(states, edgecolor='black', linewidth=0.65)

        # Drawing gridlines
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.65, 
                      color="#9d9d9d", zorder= 4, alpha=0.8)

    # Restrict labels to ONLY the bottom and left
    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = True
    gl.left_labels = True

    gl.x_inline = False
    gl.y_inline = False

    # --- ADDED: Set padding (in points) between labels and the map frame ---
    gl.xpadding = 4  # Adds space for bottom (longitude) labels
    gl.ypadding = 4  # Adds space for left (latitude) labels

    # Style the labels (removed the non-standard 'pad' key)
    gl.xlabel_style = {'size': 12, 'color': 'black', 'rotation': 15}
    gl.ylabel_style = {'size': 12, 'color': 'black'}

    # Force clean tick intervals
    gl.xlocator = mticker.FixedLocator(range(-130, -60, 10))
    gl.ylocator = mticker.FixedLocator(range(20, 55, 5))
    
    ax_box = ax.inset_axes([0.90, 0, 0.10, 0.45], zorder=5)
    ax_box.set_facecolor('white')
    ax_box.set_xticks([])
    ax_box.set_yticks([])
    for spine in ax_box.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)

    # Adjusted to end at y=0.78, leaving the top 22% of the white box for the title
    # Pushed to x=0.20 to leave plenty of room on the right side for tick labels
    cax = ax_box.inset_axes([0.10, 0.04, 0.49, 0.84]) 

    # Plot vertical colorbar (Notice: cb.ax.set_title was removed!)
    cb = plt.colorbar(im, cax=cax, orientation='vertical', ticks=custom_ticks) 

    # Style the colorbar ticks & tick labels
    cb.ax.set_yticklabels(tick_labels, fontsize=10, color='black')
    cb.ax.tick_params(direction='out', length=4, width=1)

    # --- ADDED: Text Title inside the White Box ---
    # Centered at x=0.5 (middle of the box) and y=0.89 (near the top)
    ax_box.text(
        0.5, 0.94, 
        "Days", 
        transform=ax_box.transAxes, 
        fontsize=12, 
        weight='bold', 
        color='black',
        ha='center', 
        va='center'
    )

    # Title & Layout rendering
    plt.title(title, fontsize=20, pad=13, weight='bold')
    plt.tight_layout()

    plt.savefig(save_name, dpi=300, bbox_inches='tight')
    plt.show()  

In [ ]:
from scipy import stats

gridboxes = hist_member_means[['lat', 'lon']].drop_duplicates()
results = []

for row in gridboxes.itertuples():
    lat = row.lat
    lon = row.lon

    hist_subset = hist_member_means[(hist_member_means.lat == lat) & (hist_member_means.lon == lon)]
    ftr_subset = ftr_member_means[(ftr_member_means.lat == lat) & (ftr_member_means.lon == lon)]

    hist_values = hist_subset['duration'].values
    ftr_values = ftr_subset['duration'].values

    t_stat, p_value = stats.ttest_ind(
        ftr_values,
        hist_values,
        equal_var=True,
        alternative='greater',
        nan_policy='omit'
    )
    results.append([lat, lon, p_value])

p_values_df = pd.DataFrame(results, columns=['lat', 'lon', 'p_value'])
p_values_grid = p_values_df.pivot(index='lat', columns='lon', values='p_value')

# Create p-value xarray DataArray
p_val_da = xr.DataArray(
    p_values_grid.values,
    coords=[p_values_grid.index.values, p_values_grid.columns.values],
    dims=['lat', 'lon']
)

print(p_val_da)


# 1. Pivot both ensemble means to 2D grids (lat x lon)
hist_grid = hist_ensemble_mean_duration.pivot(index='lat', columns='lon', values='mean_dur')
ftr_grid = ftr_ensemble_mean_duration.pivot(index='lat', columns='lon', values='mean_dur')

# 2. Calculate the Delta (Future - Historical)
# Because they share the exact same grid, pandas aligns the coordinates automatically
delta_duration_grid = ftr_grid - hist_grid

print(delta_duration_grid)

In [ ]:
# --- STEP 4: Apply Fractional Spatial Masking to Delta & p-values ---

# 1. Clean and sort the coordinates using .columns (longitude) and .index (latitude)
clean_lons = np.sort(np.unique(delta_duration_grid.columns.values))
clean_lats = np.sort(np.unique(delta_duration_grid.index.values))

# 2. Force regionmask to wrap the longitudes to [-180, 180] 
frac_mask_3d = us_states.mask_3D_frac_approx(clean_lons, clean_lats, wrap_lon=180)

# 3. Collapse the 'region' dimension
any_overlap_mask = (frac_mask_3d > 0).any(dim="region")

# 4. Apply the spatial mask to both DataArrays
# We reconstruct the 2D xarray DataArray using the clean index/columns
delta_da = xr.DataArray(
    delta_duration_grid.values,
    coords=[delta_duration_grid.index.values, delta_duration_grid.columns.values],
    dims=['lat', 'lon']
)

masked_delta_duration = delta_da.where(any_overlap_mask)
masked_p_values = p_val_da.where(any_overlap_mask)


# --- STEP 5: Calculate Dynamic Ticks on the Masked DELTA Grid ---
global_min = float(masked_delta_duration.min())
global_max = float(masked_delta_duration.max())

# Ensure symmetric limits for the divergent RdBu_r colorbar
limit = max(abs(global_min), abs(global_max))
combined_min_max_grid = np.array([-limit, limit])

# Generate ticks based on the intensity delta scale (not p-values!)
custom_ticks, tick_labels, exact_max, exact_min = generate_custom_ticks_05(combined_min_max_grid)


# --- STEP 6: Run Plotting Loop ---
alphas = [0.05, 0.01]

for alpha in alphas:
    # A boolean mask where p-value < alpha AND the cell is within CONUS
    significant_mask = (

            (masked_p_values<alpha)
            &
            (masked_delta_duration>0)

    )
    
    if alpha == 0.05:
        name_alpha = '005'
    else:
        name_alpha = '001'

    # Calculate total grid cells within CONUS (ignoring NaNs)
    total_gridcells = np.sum(~np.isnan(masked_delta_duration.values)) 

    # Extract lat/lon 2D grids
    lon_grid, lat_grid = np.meshgrid(masked_delta_duration['lon'].values, masked_delta_duration['lat'].values)
    
    # Extract coordinate 1D arrays for plotting stippling
    sig_mask_np = np.nan_to_num(significant_mask.values, nan=False).astype(bool)
    significant_lons = lon_grid[sig_mask_np]
    significant_lats = lat_grid[sig_mask_np]
    
    # Calculate clean CONUS percentage
    sig_cell_count = len(significant_lons) 
    percent = (sig_cell_count / total_gridcells) * 100 if total_gridcells > 0 else 0
    print(f"Alpha {alpha} - Significant cells: {sig_cell_count}/{total_gridcells} ({percent:.2f}%)")

    # Pivot and sort coordinates of the physical delta for plotting
    grid_data_delta = (
        masked_delta_duration
        .to_dataframe(name='delta')
        .reset_index()
        .pivot(index='lat', columns='lon', values='delta')
        .sort_index(ascending=True)
        .sort_index(axis=1, ascending=True)
    )

    X, Y = np.meshgrid(grid_data_delta.columns, grid_data_delta.index)

    s_lat, n_lat = 24, 52
    w_lon, e_lon = -121, -73

    crs = ccrs.LambertConformal(central_longitude=-97, central_latitude=40)

    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=crs)

    # Set map boundaries
    ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())  

    water_color = '#a5c9eb' 
    ax.set_facecolor(water_color)  

    # Plot the background physical change (delta)
    im = ax.pcolormesh(X, Y, grid_data_delta.values, 
                       transform=ccrs.PlateCarree(), 
                       cmap='RdBu_r', 
                       vmin=exact_min, vmax=exact_max, 
                       shading='auto',
                       zorder=1)
    
    if canada_geom is not None:
        ax.add_geometries([canada_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#d3d3d3', edgecolor='black', linewidth=0.8, zorder=2)
    if mexico_geom is not None:
        ax.add_geometries([mexico_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#d3d3d3', edgecolor='black', linewidth=0.8, zorder=2)
    if great_lakes_geoms:
        ax.add_geometries(great_lakes_geoms, crs=ccrs.PlateCarree(),
                          facecolor=water_color, edgecolor='black', linewidth=0.75, zorder=2)
        
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor=water_color, zorder=2)
    ax.coastlines('50m', linewidth=0.8, color='black', zorder=4)

    # USA outer outline
    usa_outline = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none',
        edgecolor='black',
        linewidth=0.8,
        zorder=4
    )
    ax.add_feature(usa_outline, zorder=3)

    # US State Boundaries
    states = cfeature.NaturalEarthFeature(
        category='cultural', 
        name='admin_1_states_provinces_lakes', 
        scale='50m', 
        facecolor='none'
    )
    ax.add_feature(states, edgecolor='black', linewidth=0.65, zorder=4)

    # Drawing gridlines (fixed zorder issue)
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.65, 
                      color="#9d9d9d", alpha=0.8, zorder=5)

    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = True
    gl.left_labels = True
    gl.x_inline = False
    gl.y_inline = False
    gl.xpadding = 4  
    gl.ypadding = 4  

    gl.xlabel_style = {'size': 12, 'color': 'black', 'rotation': 15}
    gl.ylabel_style = {'size': 12, 'color': 'black'}

    gl.xlocator = mticker.FixedLocator(range(-130, -60, 10))
    gl.ylocator = mticker.FixedLocator(range(20, 55, 5))

    # Colorbar Box Setup
    ax_box = ax.inset_axes([0.90, 0, 0.10, 0.45], zorder=6)
    ax_box.set_facecolor('white')
    ax_box.set_xticks([])
    ax_box.set_yticks([])
    for spine in ax_box.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)

    cax = ax_box.inset_axes([0.10, 0.04, 0.49, 0.84]) 
    cb = plt.colorbar(im, cax=cax, orientation='vertical', ticks=custom_ticks) 

    cb.ax.set_yticklabels(tick_labels, fontsize=10, color='black')
    cb.ax.tick_params(direction='out', length=4, width=1)

    # Text Title inside the Colorbar Box
    ax_box.text(
        0.5, 0.94, 
        'Δ Days', 
        transform=ax_box.transAxes, 
        fontsize=12, 
        weight='bold', 
        color='black',
        ha='center', 
        va='center'
    )

    # --- OVERLAY DELTA MARKERS (Stippling) ---
    # Plots the open black circles only on significant CONUS points
    ax.scatter(significant_lons, significant_lats, 
               marker='o', facecolors='none', edgecolors='black',
               s=15,          
               alpha=0.5,      
               transform=ccrs.PlateCarree(),
               zorder=1.5  # Kept above map boundaries, below gridlines
               )

    # Update title dynamically to include the computed CONUS percentage
    plt.title(f'Change in Mean Thirstwave Duration | o = p < {alpha} ({percent:.1f}% of CONUS)', 
              fontsize=16, pad=13, weight='bold')
    plt.tight_layout()

    plt.savefig(f'Duration_Mean_Delta_{name_alpha}', dpi=300, bbox_inches='tight')
    plt.show()

# Intensity Calculations/Plots

In [ ]:
daily_ds_hist = xr.open_zarr(
    "/data1/michsh/combined_thirstwave/HIST_tw_metrics.zarr"
)

daily_ds_ftr = xr.open_zarr(
    "/data1/michsh/combined_thirstwave/FTR_tw_metrics.zarr"
)

In [ ]:
time_values = daily_ds_hist.time.values

print(time_values)

# 44075

In [ ]:
# --- Safe CONUS Slicing Block ---

# 1. Take the mean over time and stack the historical dataset
intensity_hist = daily_ds_hist['tw_intensity'].mean(dim=['time'])
stacked_hist = intensity_hist.stack(ensemble_80=('AMOC', 'member'))

# Deterministically handle ascending vs descending latitude slices
lat_start, lat_end = 24, 52
if stacked_hist.lat[0] > stacked_hist.lat[-1]:
    # Latitudes are descending (North to South)
    lat_slice = slice(lat_end, lat_start)
else:
    # Latitudes are ascending (South to North)
    lat_slice = slice(lat_start, lat_end)

# Slice using the safe latitude range
conus_stacked_hist = stacked_hist.sel(lat=lat_slice, lon=slice(235, 293))

# 3. Calculate historical Mean and Sigma
hist_conus_mean = conus_stacked_hist.mean(dim='ensemble_80')
hist_conus_sigma = conus_stacked_hist.std(dim='ensemble_80', ddof=1)

# --- Repeat for the Future Dataset ---

intensity_ftr = daily_ds_ftr['tw_intensity'].mean(dim=['time'])
stacked_ftr = intensity_ftr.stack(ensemble_80=('AMOC', 'member'))

conus_stacked_ftr = stacked_ftr.sel(lat=lat_slice, lon=slice(235, 293))

# Calculate future Mean and Sigma
ftr_conus_mean = conus_stacked_ftr.mean(dim='ensemble_80')
ftr_conus_sigma = conus_stacked_ftr.std(dim='ensemble_80', ddof=1)

# --- Calculate the Delta Map ---
mean_delta_intensity = ftr_conus_mean - hist_conus_mean

test_point_hist_mean = hist_conus_mean.sel(lat=38, lon=278, method='nearest').values
test_point_ftr_mean = ftr_conus_mean.sel(lat=38, lon=278, method='nearest').values

print(hist_conus_mean.sel(lat=38, method='nearest'))
print(hist_conus_mean.sel(lon=278, method='nearest'))

print(f"CONUS Map Historical Mean at (38, 278): {test_point_hist_mean:.4f} (Expected: 1.0464)")
print(f"CONUS Map Future Mean at (38, 278):     {test_point_ftr_mean:.4f} (Expected: 1.1640)")

In [ ]:
import numpy as np

# --- STEP 1: Define Your Target Coordinates ---
hotspot_lat = 44
hotspot_lon = 264

# 250 - 276 29 
# --- STEP 2: Find the EXACT Matched Grid Coordinates First ---
# We query the coordinate metadata of the nearest point using your 2D mean map
matched_point = hist_conus_mean.sel(lat=hotspot_lat, lon=hotspot_lon, method='nearest')
exact_lat = float(matched_point.lat.values)
exact_lon = float(matched_point.lon.values)

# --- STEP 3: Extract the 80 Individual Ensemble Member Values ---
# Now we use the EXACT lat/lon values we just found to slice our 80-member stacked datasets.
# Since we have the exact coordinates, we no longer need method='nearest' here.
hist_member_vals = stacked_hist.sel(lat=exact_lat, lon=exact_lon).values
ftr_member_vals = stacked_ftr.sel(lat=exact_lat, lon=exact_lon).values

# --- STEP 4: Calculate the Statistics Across All Members ---
hist_point_mean = np.mean(hist_member_vals)
hist_point_sigma = np.std(hist_member_vals, ddof=1)

ftr_point_mean = np.mean(ftr_member_vals)
ftr_point_sigma = np.std(ftr_member_vals, ddof=1)

# --- STEP 5: Print the Results ---
print("="*65)
print(f"GRID POINT VERIFICATION FOR MIDWEST HOTSPOT")
print("="*65)
print(f"Requested Target:             ({hotspot_lat}, {hotspot_lon})")
print(f"Nearest Map Grid Coordinates: ({exact_lat:.4f}, {exact_lon:.4f})")
print("-" * 65)

print("\n--- HISTORICAL PERIOD (1850-2014) ---")
print(f"Calculated Mean (80 members): {hist_point_mean:.4f}")
print(f"Standard Deviation (Sigma):   {hist_point_sigma:.4f}")
print("Individual Member Intensity Means:")
print(np.round(hist_member_vals, 3))

print("\n" + "-"*65)

print("\n--- FUTURE PERIOD (2015-2100) ---")
print(f"Calculated Mean (80 members): {ftr_point_mean:.4f}")
print(f"Standard Deviation (Sigma):   {ftr_point_sigma:.4f}")
print("Individual Member Intensity Means:")
print(np.round(ftr_member_vals, 3))

print("\n" + "="*65)
print(f"Delta Intensity (Future - Hist): {ftr_point_mean - hist_point_mean:.4f}")
print("="*65)

In [ ]:
masked_data_arrays = []
for da in [hist_conus_mean, ftr_conus_mean]:
    # 1. Calculate the fractional overlap of each grid cell with the US States.
    # Returns a 3D dataset: [region x lat x lon]
    frac_mask_3d = us_states.mask_3D_frac_approx(da.lon, da.lat)
    
    # 2. Collapse the 'region' dimension by checking if a cell overlaps with ANY US State.
    # We set the threshold to > 0 to keep any cell that touches or sits on the border.
    any_overlap_mask = (frac_mask_3d > 0).any(dim="region")
    
    # 3. Mask the DataArray (keeping all cells where the mask is True)
    da_masked = da.where(any_overlap_mask)
    masked_data_arrays.append(da_masked)

# Reassign masked xarray DataArrays
hist_intensity_mean_masked, ftr_intensity_mean_masked = masked_data_arrays

# --- STEP 3: Calculate Limits on Masked xarray Data ---
# .min() and .max() on xarray safely ignore NaNs by default
global_min = float(min(hist_intensity_mean_masked.min(), ftr_intensity_mean_masked.min()))
global_max = float(max(hist_intensity_mean_masked.max(), ftr_intensity_mean_masked.max()))

combined_min_max_grid = np.array([global_min, global_max])
custom_ticks, tick_labels, exact_max, exact_min = generate_custom_ticks_05(combined_min_max_grid)

# --- STEP 4: Run the Plotting Loop ---
data_arrays = [hist_intensity_mean_masked, ftr_intensity_mean_masked]

for array in data_arrays: 
    plot_target_da = array  

    if plot_target_da is hist_intensity_mean_masked:
        title = "Historical Thirstwave Intensity | 80-Member Ensemble Mean"
        save_name = 'Intensity_Mean_HIST'
    elif plot_target_da is ftr_intensity_mean_masked:
        title = "Future Thirstwave Intensity | 80-Member Ensemble Mean"
        save_name = 'Intensity_Mean_FTR'

    grid_values = plot_target_da
    X, Y = np.meshgrid(plot_target_da['lon'].values, plot_target_da['lat'].values)

    s_lat, n_lat = 24, 52
    w_lon, e_lon = -121, -73

    crs = ccrs.LambertConformal(central_longitude=-97, central_latitude=40)

    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=crs)

    # Set map boundaries
    ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())  

    # -------------------------------------------------------------
    # LAYER 1: Ocean Background Color
    # -------------------------------------------------------------
    water_color = '#a5c9eb'  # Soft pastel blue
    ax.set_facecolor(water_color)  

    # -------------------------------------------------------------
    # LAYER 2: Plot the CONUS Grid Data (zorder=1)
    # -------------------------------------------------------------
    im = ax.pcolormesh(X, Y, grid_values, 
                    transform=ccrs.PlateCarree(), 
                    cmap='turbo', 
                    vmin=exact_min, vmax=exact_max, 
                    shading='auto',
                    zorder=1)

    # -------------------------------------------------------------
    # LAYER 3: Draw Solid Gray Masking over Canada & Mexico (zorder=2)
    # -------------------------------------------------------------
    if canada_geom is not None:
        ax.add_geometries([canada_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#717376', edgecolor='black', linewidth=0.65, zorder=2)
    if mexico_geom is not None:
        ax.add_geometries([mexico_geom], crs=ccrs.PlateCarree(), 
                          facecolor="#717376", edgecolor='black', linewidth=0.65, zorder=2)
    if great_lakes_geoms:
        ax.add_geometries(great_lakes_geoms, crs=ccrs.PlateCarree(),
                          facecolor=water_color, edgecolor='black', linewidth=0.65, zorder=2)

    # -------------------------------------------------------------
    # LAYER 4: Water Masking & Coastlines (zorder=3)
    # -------------------------------------------------------------
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor=water_color, zorder=2)
    ax.coastlines('50m', linewidth=0.65, color='black', zorder=3)

    # USA outer outline
    usa_outline = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none',
        edgecolor='black',
        linewidth=0.8
    )
    ax.add_feature(usa_outline, zorder=3)

    # US State Boundaries
    states = cfeature.NaturalEarthFeature(
        category='cultural', 
        name='admin_1_states_provinces_lakes', 
        scale='50m', 
        facecolor='none'
    )
    ax.add_feature(states, edgecolor='black', linewidth=0.75, zorder=3)

    point_val = float(plot_target_da.sel(lat=exact_lat, lon=exact_lon, method='nearest').values)
    print(f"{title} value at Star: {point_val:.4f}")

    # Drawing gridlines
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.65, 
                      color="#9d9d9d", zorder= 4, alpha=0.8)

    # Restrict labels to ONLY the bottom and left
    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = True
    gl.left_labels = True

    gl.x_inline = False
    gl.y_inline = False

    # --- ADDED: Set padding (in points) between labels and the map frame ---
    gl.xpadding = 4  # Adds space for bottom (longitude) labels
    gl.ypadding = 4  # Adds space for left (latitude) labels

    # Style the labels (removed the non-standard 'pad' key)
    gl.xlabel_style = {'size': 12, 'color': 'black', 'rotation': 15}
    gl.ylabel_style = {'size': 12, 'color': 'black'}

    # Force clean tick intervals
    gl.xlocator = mticker.FixedLocator(range(-130, -60, 10))
    gl.ylocator = mticker.FixedLocator(range(20, 55, 5))

        # -------------------------------------------------------------
    # LAYER 6: Inset Colorbar in the Atlantic Ocean (Bottom Right)
    # -------------------------------------------------------------
    # Create a white background box (axes coordinates are 0 to 1)
    # [x_start, y_start, width, height]
    ax_box = ax.inset_axes([0.90, 0, 0.10, 0.45], zorder=5)
    ax_box.set_facecolor('white')
    ax_box.set_xticks([])
    ax_box.set_yticks([])
    for spine in ax_box.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)

    # Adjusted to end at y=0.78, leaving the top 22% of the white box for the title
    # Pushed to x=0.20 to leave plenty of room on the right side for tick labels
    cax = ax_box.inset_axes([0.10, 0.04, 0.49, 0.84]) 

    # Plot vertical colorbar (Notice: cb.ax.set_title was removed!)
    cb = plt.colorbar(im, cax=cax, orientation='vertical', ticks=custom_ticks) 

    # Style the colorbar ticks & tick labels
    cb.ax.set_yticklabels(tick_labels, fontsize=10, color='black')
    cb.ax.tick_params(direction='out', length=4, width=1)

    # --- ADDED: Text Title inside the White Box ---
    # Centered at x=0.5 (middle of the box) and y=0.89 (near the top)
    ax_box.text(
        0.5, 0.94, 
        'mm/day', 
        transform=ax_box.transAxes, 
        fontsize=12, 
        weight='bold', 
        color='black',
        ha='center', 
        va='center'
    )

    # Title & Layout rendering
    plt.title(title, fontsize=20, pad=13, weight='bold')
    plt.tight_layout()

    plt.savefig(save_name, dpi=300, bbox_inches='tight')
    plt.show()  

In [ ]:
from scipy import stats
import xarray as xr

# 1. Run the Independent T-Test across the ensemble dimension (axis=-1)
# We use equal_var=False to perform a Welch's t-test, which is safer when variances might differ.
t_stat, p_values = stats.ttest_ind(
    conus_stacked_ftr.values, 
    conus_stacked_hist.values, 
    axis=-1, 
    equal_var=False, 
    alternative='greater',
    nan_policy='omit'
)

# 2. Wrap the resulting p-values back into an xarray DataArray matching your grid coordinates
p_val_da = xr.DataArray(
    p_values, 
    coords=[conus_stacked_hist.lat, conus_stacked_hist.lon], 
    dims=['lat', 'lon']
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Define Significance Threshold
alphas = [0.05, 0.01]

masked_data_arrays = []
da = p_val_da

# 1. Calculate the fractional overlap of each grid cell with the US States.
frac_mask_3d = us_states.mask_3D_frac_approx(da.lon, da.lat)

# 2. Collapse the 'region' dimension (True if any overlap)
any_overlap_mask = (frac_mask_3d > 0).any(dim="region")

# 3. Mask the DataArray (keeps CONUS gridpoints, sets others to NaN)
da_masked = da.where(any_overlap_mask)
masked_data_arrays.append(da_masked)

# --- FIX: Extract the actual xarray DataArray from the list ---
masked_intensity_delta = masked_data_arrays[0] 

# --- STEP 3: Calculate Limits on Masked xarray Data ---
# .min() and .max() on xarray safely ignore NaNs by default
global_min = float(masked_intensity_delta.min())
global_max = float(masked_intensity_delta.max())

combined_min_max_grid = np.array([global_min, global_max])
custom_ticks, tick_labels, exact_max, exact_min = generate_custom_ticks_05(combined_min_max_grid)

for alpha in alphas:

    if alpha == 0.05:
        name_alpha = '005'
    else:
        name_alpha = '001'

    # A boolean mask where p-value < alpha AND the cell is within CONUS (not NaN)
    significant_mask = (masked_intensity_delta < alpha)
    
    # Calculate total grid cells within CONUS (ignoring NaNs)
    total_gridcells = np.sum(~np.isnan(masked_intensity_delta.values)) 

    # Extract lat/lon 2D grids
    lon_grid, lat_grid = np.meshgrid(masked_intensity_delta['lon'].values, masked_intensity_delta['lat'].values)
    
    # Convert significant_mask to a numpy boolean array to index the meshgrids
    sig_mask_np = significant_mask.values
    # Replace any NaN-driven values in the boolean mask with False
    sig_mask_np = np.nan_to_num(sig_mask_np, nan=False).astype(bool)
    
    # Extract coordinate 1D arrays for plotting
    significant_lons = lon_grid[sig_mask_np]
    significant_lats = lat_grid[sig_mask_np]
    
    # Calculate clean CONUS percentage
    sig_cell_count = len(significant_lons) 
    percent = (sig_cell_count / total_gridcells) * 100 if total_gridcells > 0 else 0
    print(f"Alpha {alpha} - Significant cells: {sig_cell_count}/{total_gridcells} ({percent:.2f}%)")

    grid_values = mean_delta_intensity
    X, Y = np.meshgrid(mean_delta_intensity['lon'].values, mean_delta_intensity['lat'].values)

    custom_ticks, tick_labels, exact_max, exact_min = generate_custom_ticks_02(grid_values)

    s_lat, n_lat = 24, 52
    w_lon, e_lon = -121, -73

    crs = ccrs.LambertConformal(central_longitude=-97, central_latitude=40)

    fig = plt.figure(figsize=(12, 8))
    ax = plt.axes(projection=crs)

    # Set map boundaries
    ax.set_extent([w_lon, e_lon, s_lat, n_lat], crs=ccrs.PlateCarree())  

    water_color = '#a5c9eb' 
    ax.set_facecolor(water_color)  

    # Plotting the background physical grid data (using mean_delta_intensity)
    im = ax.pcolormesh(X, Y, grid_values, 
                       transform=ccrs.PlateCarree(), 
                       cmap='RdBu_r', 
                       vmin=exact_min, vmax=exact_max, 
                       shading='auto',
                       zorder=1
                       )
    
    # --- OVERLAY DELTA MARKERS (Stippling) ---
    # Plots the open black circles only on significant CONUS points
    ax.scatter(significant_lons, significant_lats, 
               marker='o', facecolors='none', edgecolors='black',
               s=15,          
               alpha=0.5,      
               transform=ccrs.PlateCarree(),
               zorder=1.5
               )
    if canada_geom is not None:
        ax.add_geometries([canada_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#d3d3d3', edgecolor='black', linewidth=0.8, zorder=2)
    if mexico_geom is not None:
        ax.add_geometries([mexico_geom], crs=ccrs.PlateCarree(), 
                          facecolor='#d3d3d3', edgecolor='black', linewidth=0.8, zorder=2)
        
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor=water_color, zorder=3)
    ax.coastlines('50m', linewidth=0.8, color='black', zorder=4)

    # USA outer outline
    usa_outline = cfeature.NaturalEarthFeature(
        category='cultural',
        name='admin_0_boundary_lines_land',
        scale='50m',
        facecolor='none',
        edgecolor='black',
        linewidth=0.8
    )
    ax.add_feature(usa_outline, zorder=4)

    # US State Boundaries
    states = cfeature.NaturalEarthFeature(
        category='cultural', 
        name='admin_1_states_provinces_lakes', 
        scale='50m', 
        facecolor='none'
    )
    ax.add_feature(states, edgecolor='black', linewidth=0.65, zorder=4)

    # Drawing gridlines
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, linewidth=0.65, 
                      color="#9d9d9d", zorder= 5, alpha=0.8)

    # Restrict labels to ONLY the bottom and left
    gl.top_labels = False
    gl.right_labels = False
    gl.bottom_labels = True
    gl.left_labels = True

    gl.x_inline = False
    gl.y_inline = False

    # --- ADDED: Set padding (in points) between labels and the map frame ---
    gl.xpadding = 4  # Adds space for bottom (longitude) labels
    gl.ypadding = 4  # Adds space for left (latitude) labels

    # Style the labels (removed the non-standard 'pad' key)
    gl.xlabel_style = {'size': 12, 'color': 'black', 'rotation': 15}
    gl.ylabel_style = {'size': 12, 'color': 'black'}

    # Force clean tick intervals
    gl.xlocator = mticker.FixedLocator(range(-130, -60, 10))
    gl.ylocator = mticker.FixedLocator(range(20, 55, 5))

    # Colorbar Box Setup
    ax_box = ax.inset_axes([0.90, 0, 0.10, 0.45], zorder=6)
    ax_box.set_facecolor('white')
    ax_box.set_xticks([])
    ax_box.set_yticks([])
    for spine in ax_box.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor('black')
        spine.set_linewidth(1.0)

    cax = ax_box.inset_axes([0.10, 0.04, 0.49, 0.84]) 
    cb = plt.colorbar(im, cax=cax, orientation='vertical', ticks=custom_ticks) 

    cb.ax.set_yticklabels(tick_labels, fontsize=10, color='black')
    cb.ax.tick_params(direction='out', length=4, width=1)

    # Text Title inside the White Box
    ax_box.text(
        0.5, 0.94, 
        'Δ mm/day', 
        transform=ax_box.transAxes, 
        fontsize=12, 
        weight='bold', 
        color='black',
        ha='center', 
        va='center'
    )


    # Update title dynamically to include the computed CONUS percentage
    plt.title(f'Change in Mean Thirstwave Intensity | o = p < {alpha} ({percent:.1f}% of CONUS)', 
              fontsize=16, pad=13, weight='bold')
    plt.tight_layout()

    plt.savefig(f'Intensity_Mean_Delta_{name_alpha}', dpi=300, bbox_inches='tight')
    plt.show()